# Exercises: Hypothesis Testing

**A Waiter's Tips**

The following description was retrieved from Kaggle page.

> Food servers’ tips in restaurants may be influenced by many factors, including the nature of the restaurant, size of the party, and table locations in the restaurant. Restaurant managers need to know which factors matter when they assign tables to food servers. For the sake of staff morale, they usually want to avoid either the substance or the appearance of unfair treatment of the servers, for whom tips (at least in restaurants in the United States) are a major component of pay. In one restaurant, a food server recorded the following data on all customers they served during an interval of two and a half months in early 1990. The restaurant, located in a suburban shopping mall, was part of a national chain and served a varied menu. In observance of local law, the restaurant offered to seat in a non-smoking section to patrons who requested it. Each record includes a day and time, and taken together, they show the server’s work schedule.

Acknowledgements The data was reported in a collection of case studies for business statistics. Bryant, P. G. and Smith, M (1995) Practical Data Analysis: Case Studies in Business Statistics. Homewood, IL: Richard D. Irwin Publishing

The dataset is also available through the Python package Seaborn.

In [1]:
import seaborn as sns

tips = sns.load_dataset("tips")

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf

tips = tips.copy()
tips["tip_pct"] = tips["tip"] / tips["total_bill"]

tips.head()

ModuleNotFoundError: No module named 'statsmodels'

Here's a description of each column in the dataset:

- `total_bill`: The total bill amount, including the cost of food and drinks.
- `tip`: The tip amount given by the customer.
- `sex`: The gender of the customer (e.g., Male or Female).
- `smoker`: Whether the customer is a smoker or not (e.g., Yes or No).
- `day`: The day of the week when the transaction occurred (e.g., Sun, Sat, Thu, etc.).
- `time`: The time of day when the transaction occurred, typically categorized as Lunch or Dinner.
- `size`: The size of the party or group of customers.

**Your Task**: is to accept or reject the following hypothesis using statistical testing:

- Hypothesis $H_1$: smoking is associated with time of visit
- Hypothesis $H_2$: the bigger the group the higher the tip
- Hypothesis $H_3$: group size is different based on the time of visit
- Hypothesis $H_4$: (... come up with a hypothesis of your own ...)
- Finally, analyze if size (party size) is a **confounder**. That is, does a larger party cause a higher tip, or simply a higher bill which then leads to a higher tip?

---

## H1: Smoking is associated with time of visit

To test whether smoking status is associated with the time of visit, we use a **chi-square test of independence** because both variables are categorical.

- **Null hypothesis (H0):** smoking status and time of visit are independent
- **Alternative hypothesis (H1):** smoking status and time of visit are associated

In [ ]:
table_h1 = pd.crosstab(tips["smoker"], tips["time"])
chi2, p_value_h1, dof, expected = stats.chi2_contingency(table_h1)

print("Contingency table:")
print(table_h1)
print("\nChi-square statistic:", chi2)
print("p-value:", p_value_h1)
print("Degrees of freedom:", dof)
print("\nExpected frequencies:")
print(pd.DataFrame(expected, index=table_h1.index, columns=table_h1.columns))

Contingency table:
time    Lunch  Dinner
smoker               
Yes        23      70
No         45     106

Chi-square statistic: 0.5053733928754355
p-value: 0.4771485672079724
Degrees of freedom: 1

Expected frequencies:
time        Lunch      Dinner
smoker                       
Yes     25.918033   67.081967
No      42.081967  108.918033


### Decision for H1

The p-value is greater than 0.05, so we fail to reject the null hypothesis.

**Conclusion:** there is not enough evidence to say that smoking status is associated with the time of visit.

## H2: The bigger the group, the higher the tip

To test whether larger groups tend to leave higher tips, we measure the relationship between `size` and `tip`.

Because `size` is a discrete variable and the relationship may not be perfectly linear, we use **Spearman correlation**.
- **H0:** there is no association between group size and tip
- **H1:** larger groups are associated with higher tips

In [ ]:
corr_h2, p_value_h2 = stats.spearmanr(tips["size"], tips["tip"])

print("Spearman correlation:", corr_h2)
print("p-value:", p_value_h2)

Spearman correlation: 0.46826792926211475
p-value: 1.059850307726079e-14


### Decision for H2

The correlation is positive and the p-value is much smaller than 0.05.

**Conclusion:** we reject the null hypothesis and accept that bigger groups are associated with higher tips.

## H3: Group size is different based on the time of visit

We compare party size between Lunch and Dinner.

Because `size` is discrete and not strongly normal, we use the **Mann–Whitney U test**.
- **H0:** the distribution of group size is the same for Lunch and Dinner
- **H1:** the distribution of group size is different for Lunch and Dinner

In [ ]:
lunch_size = tips.loc[tips["time"] == "Lunch", "size"]
dinner_size = tips.loc[tips["time"] == "Dinner", "size"]

u_stat, p_value_h3 = stats.mannwhitneyu(lunch_size, dinner_size, alternative="two-sided")

print("Lunch mean size:", lunch_size.mean())
print("Dinner mean size:", dinner_size.mean())
print("Lunch median size:", lunch_size.median())
print("Dinner median size:", dinner_size.median())
print("\nMann-Whitney U statistic:", u_stat)
print("p-value:", p_value_h3)

Lunch mean size: 2.411764705882353
Dinner mean size: 2.6306818181818183
Lunch median size: 2.0
Dinner median size: 2.0

Mann-Whitney U statistic: 4897.0
p-value: 0.010166655373207559


### Decision for H3

The p-value is below 0.05.

**Conclusion:** we reject the null hypothesis and conclude that party size differs by time of visit. Dinner parties tend to be slightly larger on average.

## H4: Total bill is different based on the time of visit

My own hypothesis is that the total bill differs between Lunch and Dinner.

We compare `total_bill` between the two groups using the **Mann–Whitney U test**.
- **H0:** the distribution of total bill is the same for Lunch and Dinner
- **H1:** the distribution of total bill is different for Lunch and Dinner

In [ ]:
lunch_bill = tips.loc[tips["time"] == "Lunch", "total_bill"]
dinner_bill = tips.loc[tips["time"] == "Dinner", "total_bill"]

u_stat_h4, p_value_h4 = stats.mannwhitneyu(lunch_bill, dinner_bill, alternative="two-sided")

print("Lunch mean total bill:", lunch_bill.mean())
print("Dinner mean total bill:", dinner_bill.mean())
print("\nMann-Whitney U statistic:", u_stat_h4)
print("p-value:", p_value_h4)

Lunch mean total bill: 17.168676470588235
Dinner mean total bill: 20.79715909090909

Mann-Whitney U statistic: 4380.5
p-value: 0.0011832506228466827


### Decision for H4

The p-value is below 0.05.

**Conclusion:** we reject the null hypothesis and conclude that total bill differs by time of visit. Dinner bills are generally higher than Lunch bills.

## Confounder Analysis: Is party size a confounder?

To understand whether party size directly affects the tip, or whether it works mainly through a higher total bill, we compare:

1. A model with `tip ~ size`
2. A model with `tip ~ total_bill`
3. A model with `tip ~ total_bill + size`

If the effect of `size` becomes much weaker after controlling for `total_bill`, then total bill explains a large part of the relationship.

In [ ]:
model_size = smf.ols("tip ~ size", data=tips).fit()
model_bill = smf.ols("tip ~ total_bill", data=tips).fit()
model_both = smf.ols("tip ~ total_bill + size", data=tips).fit()

print("Model 1: tip ~ size")
print(model_size.params)
print(model_size.pvalues)
print("R-squared:", model_size.rsquared)

print("\nModel 2: tip ~ total_bill")
print(model_bill.params)
print(model_bill.pvalues)
print("R-squared:", model_bill.rsquared)

print("\nModel 3: tip ~ total_bill + size")
print(model_both.params)
print(model_both.pvalues)
print("R-squared:", model_both.rsquared)

NameError: name 'smf' is not defined

In [ ]:
print("Correlation between size and total_bill:", tips["size"].corr(tips["total_bill"]))
print("Correlation between size and tip:", tips["size"].corr(tips["tip"]))
print("Correlation between total_bill and tip:", tips["total_bill"].corr(tips["tip"]))

Correlation between size and total_bill: 0.5983151309049018
Correlation between size and tip: 0.4892987752303579
Correlation between total_bill and tip: 0.6757341092113645


In [ ]:
# Optional: look at tip percentage instead of raw tip
model_tip_pct = smf.ols("tip_pct ~ size + total_bill", data=tips).fit()

print(model_tip_pct.params)
print(model_tip_pct.pvalues)
print("R-squared:", model_tip_pct.rsquared)

NameError: name 'smf' is not defined

## Final Summary

- **H1:** Rejected the alternative. Smoking status is not significantly associated with time of visit.
- **H2:** Accepted. Bigger groups are associated with higher tips.
- **H3:** Accepted. Group size differs by time of visit.
- **H4:** Accepted. Total bill differs by time of visit, with Dinner generally higher than Lunch.
- **Confounder analysis:** Party size affects tip partly because larger groups usually produce larger total bills. Total bill explains much of the size–tip relationship.